# **Installation**

In [3]:
pip install pandas

In [4]:
import pandas as pd
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

In [10]:
!pip install duckdb duckdb-engine jupysql --quiet

import duckdb
%load_ext sql
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

In [33]:
#Once csv loaded in Collab - run this cell

# 1. Connexion à une base en mémoire
conn = duckdb.connect(database=':memory:')

# 2. Création de table SQL à partir des CSV
conn.execute("CREATE TABLE dropout AS SELECT * FROM read_csv_auto('clinical_trial_dataset_1000(1).csv')")

# 3. Configurer JupySQL
%reload_ext sql
%config SqlMagic.autopandas = True
%sql conn

In [5]:
# 1. CRÉER BASE SQLITE directement dans Colab
import sqlite3
import pandas as pd

# Ta requête SQL → directement DataFrame
query = """
SELECT
  Gender, Age, Health_Condition, Trial_Phase, previous_adherence_score, Dropout_Flag
FROM clinical_dropout.clinical_trial_dataset_1000(1)
"""

# Si tu as accès BigQuery
print("Méthode 1 : SQLite en mémoire - Prêt !")

Méthode 1 : SQLite en mémoire - Prêt !


In [1]:
!pip install pandas scikit-learn joblib

##upload CSV file

In [2]:
from google.colab import files
uploaded = files.upload()

Saving clinical_trial_dataset_1000(1).csv to clinical_trial_dataset_1000(1) (1).csv


In [8]:
import pandas as pd

df = pd.read_csv("clinical_trial_dataset_1000(1).csv")
print("Dataset loaded")
df.head()

Dataset loaded


,Participant_ID,Age,Gender,Ethnicity,Health_Condition,Comorbidities,Previous_Adherence_Score,Trial_Phase,Dropout_Flag,Dropout_Reason
0,P0001,58,Male,White,NaN,Arthritis,0.85,Phase I,No,NaN
1,P0002,32,Male,White,NaN,NaN,0.72,Phase I,No,NaN
2,P0003,55,Female,White,Diabetes,Obesity,0.70,Phase I,Yes,Lack of Improvement
3,P0004,50,Male,White,Asthma,Depression,0.70,Phase III,No,NaN
4,P0005,59,Female,Black,Heart Disease,Depression,1.00,Phase II,Yes,Medical Advice


# **SQL Coding**

In [34]:
%%sql
SELECT * FROM dropout

Running query in 'DuckDBPyConnection'

,Participant_ID,Age,Gender,Ethnicity,Health_Condition,Comorbidities,Previous_Adherence_Score,Trial_Phase,Dropout_Flag,Dropout_Reason
0,P0001,58,Male,White,None,Arthritis,0.85,Phase I,False,None
1,P0002,32,Male,White,None,None,0.72,Phase I,False,None
2,P0003,55,Female,White,Diabetes,Obesity,0.70,Phase I,True,Lack of Improvement
3,P0004,50,Male,White,Asthma,Depression,0.70,Phase III,False,None
4,P0005,59,Female,Black,Heart Disease,Depression,1.00,Phase II,True,Medical Advice
...,...,...,...,...,...,...,...,...,...,...
995,P0996,44,Male,Asian,None,None,0.72,Phase I,True,Medical Advice
996,P0997,54,Female,Asian,Hypertension,Arthritis,0.76,Phase II,True,Logistical Issues
997,P0998,61,Male,Hispanic,None,Kidney Disease,0.98,Phase I,False,None
998,P0999,42,Male,Asian,Heart Disease,Depression,0.56,Phase I,True,None


In [124]:
%%sql

-- Total Global Dropout Participants
SELECT DISTINCT
COUNT(*) AS Total_participants,
SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout, -- dropout = 1
CONCAT(ROUND(100*SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),2),'%') As dropout_rate
FROM dropout

Running query in 'DuckDBPyConnection'

,Total_participants,nb_dropout,dropout_rate
0,1000,262.0,26.2%


In [158]:
%%sql

-- Dropout by Gender
  SELECT DISTINCT
    Gender,
    COUNT(*) AS nb_participants,
    SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout,
    CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2),'%') AS Dropout_rate,
    DENSE_RANK() OVER (ORDER BY 100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS rank
  FROM dropout
  GROUP BY Gender,
  ORDER BY rank

  -- on remarque 2x plus de femmes participantes (616 vs 384) => Possibilité de l'impact du déséquilibre de la taille d’échantillon 'Female Gender' à surveiller... si regression logistique utiliser une pondération

Running query in 'DuckDBPyConnection'

,Gender,nb_participants,nb_dropout,Dropout_rate,rank
0,Male,384,101.0,26.3%,1
1,Female,616,161.0,26.14%,2


In [144]:
%%sql

-- By Age
WITH ranked_dropout AS (
  SELECT
    Case FLOOR(Age/10)
    WHEN 1 THEN '10-19'
    WHEN 2 THEN '20-29'
    WHEN 3 THEN '30-39'
    WHEN 4 THEN '40-49'
    WHEN 5 THEN '50-59'
    WHEN 6 THEN '60-69'
    WHEN 7 THEN '70-79'
    WHEN 8 THEN '80-89'
    ELSE '90+'
    END AS age_group,
    COUNT(*) AS nb_participants,
    SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout,
    CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2),'%') AS dropout_rate,
    DENSE_RANK() OVER (ORDER BY 100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS rank
  FROM dropout
  GROUP BY CASE FLOOR(Age/10) WHEN 1 THEN '10-19' WHEN 2 THEN '20-29' WHEN 3 THEN '30-39' WHEN 4 THEN '40-49' WHEN 5 THEN '50-59' WHEN 6 THEN '60-69' WHEN 7 THEN '70-79' WHEN 8 THEN '80-89' ELSE '90+' END
)
SELECT
  rank,
  age_group,
  nb_participants,
  nb_dropout,
  dropout_rate
FROM ranked_dropout
ORDER BY rank ASC

Running query in 'DuckDBPyConnection'

,rank,age_group,nb_participants,nb_dropout,dropout_rate
0,1,80-89,19,8.0,42.11%
1,2,20-29,158,46.0,29.11%
2,3,10-19,18,5.0,27.78%
3,4,50-59,160,42.0,26.25%
4,5,40-49,165,42.0,25.45%
5,6,70-79,185,47.0,25.41%
6,7,60-69,153,38.0,24.84%
7,8,30-39,142,34.0,23.94%


In [157]:
%%sql

-- By Health Condition and Comorbidities
WITH ranked_dropout AS (
  SELECT
    Health_Condition,
    Comorbidities,
    COUNT(*) AS nb_participants,
    SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout,
    CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2),'%') AS Dropout_rate,
    DENSE_RANK() OVER (ORDER BY 100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS rank
  FROM dropout
  GROUP BY Health_Condition, Comorbidities
)
SELECT
  rank,
  Health_Condition,
  Comorbidities,
  nb_participants,
  nb_dropout,
  Dropout_rate,
FROM ranked_dropout
ORDER BY rank
LIMIT 3

Running query in 'DuckDBPyConnection'

,rank,Health_Condition,Comorbidities,nb_participants,nb_dropout,Dropout_rate
0,1,None,None,32,13.0,40.63%
1,2,Asthma,Arthritis,40,16.0,40.0%
2,3,Hypertension,Arthritis,38,15.0,39.47%


In [161]:
%%sql

-- By Gender/ Age / Health Condition and Comorbidities
WITH ranked_dropout AS (
  SELECT
    gender,
    Comorbidities,
    Case FLOOR(Age/10)
    WHEN 1 THEN '10-19'
    WHEN 2 THEN '20-29'
    WHEN 3 THEN '30-39'
    WHEN 4 THEN '40-49'
    WHEN 5 THEN '50-59'
    WHEN 6 THEN '60-69'
    WHEN 7 THEN '70-79'
    WHEN 8 THEN '80-89'
    ELSE '90+'
    END AS age_group,
    Health_Condition,
    COUNT(*) AS nb_participants,
    SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout,
    CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2),'%') AS dropout_rate,
    DENSE_RANK() OVER (ORDER BY 100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS rank
  FROM dropout
  GROUP BY gender, Comorbidities, CASE FLOOR(Age/10) WHEN 1 THEN '10-19' WHEN 2 THEN '20-29' WHEN 3 THEN '30-39' WHEN 4 THEN '40-49' WHEN 5 THEN '50-59' WHEN 6 THEN '60-69' WHEN 7 THEN '70-79' WHEN 8 THEN '80-89' ELSE '90+' END, Health_Condition
  HAVING COUNT(*) >= 2
)
SELECT
  rank,
  gender,
  age_group,
  Comorbidities,
  Health_Condition,
  nb_participants,
  nb_dropout,
  dropout_rate
FROM ranked_dropout
ORDER BY rank ASC
LIMIT 15

Running query in 'DuckDBPyConnection'

,rank,Gender,age_group,Comorbidities,Health_Condition,nb_participants,nb_dropout,dropout_rate
0,1,Female,80-89,Depression,Hypertension,2,2.0,100.0%
1,1,Male,50-59,Obesity,None,2,2.0,100.0%
2,1,Female,30-39,Arthritis,Hypertension,4,4.0,100.0%
3,1,Male,20-29,Depression,Asthma,3,3.0,100.0%
4,1,Male,40-49,None,None,3,3.0,100.0%
5,1,Male,40-49,Depression,Diabetes,2,2.0,100.0%
6,2,Female,40-49,Kidney Disease,Asthma,4,3.0,75.0%
7,2,Female,70-79,Depression,Cancer,4,3.0,75.0%
8,2,Male,30-39,Obesity,Cancer,4,3.0,75.0%
9,2,Male,40-49,Kidney Disease,Asthma,4,3.0,75.0%


# Dropout Phases Analysis (PHASE 1, PHASE 2 or PHASE 3)

In [155]:
%%sql

-- ALL PARTICIPANTS
  SELECT DISTINCT
    Trial_Phase,
    COUNT(*) AS nb_participants,
    SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout,
    CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2),'%') AS dropout_rate,
    DENSE_RANK() OVER (ORDER BY 100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS rank
  FROM dropout
  GROUP BY Trial_Phase
  ORDER BY rank

Running query in 'DuckDBPyConnection'

,Trial_Phase,nb_participants,nb_dropout,dropout_rate,rank
0,Phase I,326,87.0,26.69%,1
1,Phase III,306,81.0,26.47%,2
2,Phase II,368,94.0,25.54%,3


In [166]:
%%sql

SELECT DISTINCT
Trial_Phase,
Gender,
Health_Condition,
Comorbidities,
Case FLOOR(Age/10)
    WHEN 1 THEN '10-19'
    WHEN 2 THEN '20-29'
    WHEN 3 THEN '30-39'
    WHEN 4 THEN '40-49'
    WHEN 5 THEN '50-59'
    WHEN 6 THEN '60-69'
    WHEN 7 THEN '70-79'
    WHEN 8 THEN '80-89'
    ELSE '90+'
    END AS age_group,
COUNT(*) AS nb_participants,
SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout,
CONCAT(ROUND(100*SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),2),'%') AS dropout_rate,
DENSE_RANK () OVER (ORDER BY 100*SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC) AS rank
FROM dropout
GROUP BY Trial_Phase, Gender,  Comorbidities, Health_Condition, CASE FLOOR(Age/10) WHEN 1 THEN '10-19' WHEN 2 THEN '20-29' WHEN 3 THEN '30-39' WHEN 4 THEN '40-49' WHEN 5 THEN '50-59' WHEN 6 THEN '60-69' WHEN 7 THEN '70-79' WHEN 8 THEN '80-89' ELSE '90+' END
HAVING COUNT(*) >= 2
ORDER BY rank
LIMIT 10

Running query in 'DuckDBPyConnection'

,Trial_Phase,Gender,Health_Condition,Comorbidities,age_group,nb_participants,nb_dropout,dropout_rate,rank
0,Phase I,Male,Asthma,Depression,20-29,2,2.0,100.0%,1
1,Phase III,Male,Heart Disease,Kidney Disease,70-79,2,2.0,100.0%,1
2,Phase II,Female,Diabetes,Kidney Disease,70-79,2,2.0,100.0%,1
3,Phase I,Male,Cancer,Arthritis,30-39,2,2.0,100.0%,1
4,Phase I,Male,Cancer,Obesity,30-39,2,2.0,100.0%,1
5,Phase I,Female,Hypertension,Arthritis,30-39,3,3.0,100.0%,1
6,Phase III,Female,None,Arthritis,20-29,2,2.0,100.0%,1
7,Phase II,Male,Hypertension,Obesity,30-39,2,2.0,100.0%,1
8,Phase I,Male,None,None,40-49,2,2.0,100.0%,1
9,Phase II,Female,Asthma,Arthritis,20-29,2,2.0,100.0%,1


In [142]:
%%sql

-- ZOOM INTO PHASE 3/ 20-29 YEARS FEMALE PARTICIPANTS BY HEALTH CONDITION
SELECT DISTINCT
  'Phase III 20-29 years' AS target_population,
  Health_Condition,
  Comorbidities,
  COUNT(*) AS total_participants,
  SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS total_dropouts,
  CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2), '%') AS dropout_rate
FROM dropout
WHERE Trial_Phase = 'Phase III'
  AND Age BETWEEN 20 AND 29
  AND Gender = 'Female'
GROUP BY Health_Condition, Comorbidities
ORDER BY 100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) DESC

Running query in 'DuckDBPyConnection'

,target_population,Health_Condition,Comorbidities,total_participants,total_dropouts,dropout_rate
0,Phase III 20-29 years,Cancer,Kidney Disease,1,1.0,100.0%
1,Phase III 20-29 years,None,Arthritis,2,2.0,100.0%
2,Phase III 20-29 years,Asthma,None,1,1.0,100.0%
3,Phase III 20-29 years,None,Kidney Disease,1,1.0,100.0%
4,Phase III 20-29 years,None,Depression,1,1.0,100.0%
5,Phase III 20-29 years,Heart Disease,None,3,1.0,33.33%
6,Phase III 20-29 years,Asthma,Depression,1,0.0,0.0%
7,Phase III 20-29 years,Diabetes,Kidney Disease,1,0.0,0.0%
8,Phase III 20-29 years,Hypertension,Depression,1,0.0,0.0%
9,Phase III 20-29 years,Heart Disease,Arthritis,2,0.0,0.0%


In [186]:
%%sql

--1. Vérification de la plage réelle des scores
SELECT DISTINCT
ROUND(AVG(previous_adherence_score), 3) AS avg_adherence_global,
MIN(previous_adherence_score) AS min_score,
MAX(previous_adherence_score) AS max_score,
COUNT(*) AS total_rows
FROM dropout

Running query in 'DuckDBPyConnection'

,avg_adherence_global,min_score,max_score,total_rows
0,0.75,0.1,1.0,1000


In [202]:
%%sql

--TOP 10 risks profils counting adherence_score

WITH risk_with_adherence AS (
  SELECT
    Gender,
    CASE FLOOR(Age/10)
    WHEN 1 THEN '10-19'
    WHEN 2 THEN '20-29'
    WHEN 3 THEN '30-39'
    WHEN 4 THEN '40-49'
    WHEN 5 THEN '50-59'
    WHEN 6 THEN '60-69'
    WHEN 7 THEN '70-79'
    WHEN 8 THEN '80-89'
    ELSE '90+'
    END AS age_group,
    Health_Condition,
    Comorbidities,
    Trial_Phase,
    AVG(previous_adherence_score) AS Avg_historic_adherence,
    COUNT(*) AS nb_participants,
    SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) AS nb_dropout,
    CONCAT(ROUND(100.0 * SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2),'%') AS dropout_rate,

    -- NOUVEAU SCORE COMBINÉ (Random Forest amélioré)
    DENSE_RANK() OVER (ORDER BY (1-AVG(previous_adherence_score))*30 +  -- Mauvaise adhérence passée = +risque
      (SUM(CASE WHEN Dropout_Flag = 'Yes' THEN 1 ELSE 0 END)/COUNT(*))*70 DESC) AS rank -- Dropout actuel
  FROM dropout
  GROUP BY Comorbidities, Gender, Health_Condition, Trial_Phase, CASE FLOOR(Age/10) WHEN 1 THEN '10-19' WHEN 2 THEN '20-29' WHEN 3 THEN '30-39' WHEN 4 THEN '40-49' WHEN 5 THEN '50-59' WHEN 6 THEN '60-69' WHEN 7 THEN '70-79' WHEN 8 THEN '80-89' ELSE '90+' END
  HAVING COUNT(*) >= 1
  AND AVG(previous_adherence_score) < 0.6
)
SELECT
  rank, Gender, age_group, Health_Condition, Trial_Phase, Comorbidities, dropout_rate,
  ROUND(Avg_historic_adherence, 3) AS Avg_historic_adherence,
  CASE
    WHEN Avg_historic_adherence BETWEEN 0 AND 0.4 THEN '🚨 HIGHT'
    WHEN Avg_historic_adherence BETWEEN 0.41 AND 0.7 THEN '⚠️  MEDIUM'
    ELSE '✅ LOW'
  END AS adherence_status
FROM risk_with_adherence
ORDER BY rank
LIMIT 6

Running query in 'DuckDBPyConnection'

,rank,Gender,age_group,Health_Condition,Trial_Phase,Comorbidities,dropout_rate,Avg_historic_adherence,adherence_status
0,1,Female,70-79,Heart Disease,Phase I,Depression,100.0%,0.34,🚨 HIGHT
1,2,Male,60-69,Asthma,Phase III,Kidney Disease,100.0%,0.35,🚨 HIGHT
2,3,Female,70-79,Hypertension,Phase III,Arthritis,100.0%,0.37,🚨 HIGHT
3,4,Female,70-79,Asthma,Phase III,None,100.0%,0.40,🚨 HIGHT
4,5,Female,60-69,Asthma,Phase III,None,100.0%,0.43,⚠️ MEDIUM
5,5,Female,40-49,None,Phase II,Depression,100.0%,0.43,⚠️ MEDIUM
